# 01 - Data Overview & EDA + Evaluation

**Person 1**: تجهيز الـ datasets + EDA + Evaluation للـ Supervised


## 1. Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# For evaluation
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

pd.set_option('mode.chained_assignment', None)


## 2. Load Raw Dataset


In [ ]:
# Load classification dataset
df_classification = pd.read_csv('data/raw/dirty_cafe_sales.csv')

# Load clustering dataset
df_clustering = pd.read_csv('data/raw/retail_store_sales.csv')


## 3. Dataset Overview - Classification


In [ ]:
print("Classification Dataset Shape:", df_classification.shape)
display(df_classification.head())
display(df_classification.info())


## 4. Missing Values Overview


In [ ]:
missing_overview = df_classification.isnull().sum()
missing_percent = (missing_overview / len(df_classification)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_overview,
    'Percentage': missing_percent
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
display(missing_df)


## 5. Duplicates Overview


In [ ]:
duplicate_count = df_classification.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")
print(f"Percentage of duplicates: {(duplicate_count / len(df_classification)) * 100:.2f}%")


## 6. Basic Visualizations


In [ ]:
# Visualization 1: Distribution of Total Spent
df_numeric = df_classification.copy()
df_numeric['Total Spent'] = pd.to_numeric(df_numeric['Total Spent'], errors='coerce')
df_numeric = df_numeric.dropna(subset=['Total Spent'])

plt.figure(figsize=(10, 5))
plt.hist(df_numeric['Total Spent'], bins=50, edgecolor='black')
plt.xlabel('Total Spent')
plt.ylabel('Frequency')
plt.title('Distribution of Total Spent')
plt.show()


In [ ]:
# Visualization 2: Scatter plot - Quantity vs Total Spent
df_numeric['Quantity'] = pd.to_numeric(df_numeric['Quantity'], errors='coerce')
df_numeric = df_numeric.dropna(subset=['Quantity', 'Total Spent'])

plt.figure(figsize=(10, 5))
plt.scatter(df_numeric['Quantity'], df_numeric['Total Spent'], alpha=0.6, color='purple')
plt.xlabel('Quantity')
plt.ylabel('Total Spent')
plt.title('Quantity vs Total Spent')
plt.show()


In [ ]:
# Visualization 3: Correlation Matrix
numeric_cols = df_numeric.select_dtypes(include=[np.number])
numeric_cols = numeric_cols.dropna()

if not numeric_cols.empty and numeric_cols.shape[1] > 1:
    plt.figure(figsize=(8, 6))
    sns.heatmap(numeric_cols.corr(), annot=True, fmt=".2f", cmap='viridis')
    plt.title('Correlation Matrix')
    plt.show()
else:
    print("Warning: No sufficient numeric columns for correlation matrix.")


## 7. Evaluation - Supervised Models

**Note**: This section evaluates the supervised models after they are trained by Person 6.
Load the predictions from the trained models and evaluate them.


In [ ]:
# Load preprocessed data (from Person 5)
X_test = pd.read_csv('data/processed/X_test_classification.csv')
y_test = pd.read_csv('data/processed/y_test_classification.csv')

# Load model predictions (from Person 6)
# Note: Person 6 should save predictions as CSV files
# Example: y_pred_lr.csv, y_pred_nb.csv, y_pred_dt.csv, y_pred_svm.csv

try:
    y_pred_lr = pd.read_csv('data/processed/y_pred_lr.csv').values.flatten()
    y_pred_nb = pd.read_csv('data/processed/y_pred_nb.csv').values.flatten()
    y_pred_dt = pd.read_csv('data/processed/y_pred_dt.csv').values.flatten()
    y_pred_svm = pd.read_csv('data/processed/y_pred_svm.csv').values.flatten()
    
    y_test = y_test.values.flatten()
    
    models = {
        'Logistic Regression': y_pred_lr,
        'Naive Bayes': y_pred_nb,
        'Decision Tree': y_pred_dt,
        'SVM': y_pred_svm
    }
    
    # Calculate metrics for each model
    results = []
    for model_name, y_pred in models.items():
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        results.append({
            'Model': model_name,
            'Accuracy': accuracy,
            'F1 Score': f1
        })
    
    results_df = pd.DataFrame(results)
    display(results_df)
    
except FileNotFoundError:
    print("Model predictions not found. Please run Person 6's notebook first to generate predictions.")
    print("Expected files:")
    print("- data/processed/y_pred_lr.csv")
    print("- data/processed/y_pred_nb.csv")
    print("- data/processed/y_pred_dt.csv")
    print("- data/processed/y_pred_svm.csv")


## 8. Confusion Matrix Visualization


In [ ]:
try:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    
    for idx, (model_name, y_pred) in enumerate(models.items()):
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx])
        axes[idx].set_title(f'Confusion Matrix - {model_name}')
        axes[idx].set_xlabel('Predicted')
        axes[idx].set_ylabel('Actual')
    
    plt.tight_layout()
    plt.show()
    
except NameError:
    print("Models dictionary not found. Please run the evaluation cell above first.")


## 9. Classification Report


In [ ]:
try:
    for model_name, y_pred in models.items():
        print(f"\n{'='*50}")
        print(f"Classification Report - {model_name}")
        print(f"{'='*50}")
        print(classification_report(y_test, y_pred))
        
except NameError:
    print("Models dictionary not found. Please run the evaluation cell above first.")
